# Cumulative Spend Distribution Analysis

This notebook characterizes the underlying cumulative spend-position distribution that feeds the later Beta CDF modeling work. It should be read before `clustered_curve_beta_cdf_model.ipynb`: this page explains the empirical shape of `CUMULATIVEBURNPCT` as a function of `ELAPSEDPCT`, while the later notebook evaluates predictive reference curves against held-out items.

## Dataset and Overall Character

| Metric | Value |
| --- | --- |
| Input CSV | clustered_data_input.csv |
| Cluster curve rows | 7,631 |
| Items | 875 |
| Train rows used for distribution fitting | 6,214 |
| Train items | 701 |
| Mean cumulative spend pct | 62.18% |
| Median cumulative spend pct | 66.69% |
| Std dev cumulative spend pct | 32.07% |
| Share at completed edge near 100% | 13.27% |
| Share at zero edge near 0% | 0.16% |
| Pearson corr elapsed vs cumulative spend | 0.7553 |
| Spearman corr elapsed vs cumulative spend | 0.7817 |

In [ ]:
import csv
from pathlib import Path

path = Path('clustered_data_input.csv')
with path.open(newline='', encoding='utf-8-sig') as handle:
    reader = csv.DictReader(handle)
    rows = list(reader)
print(path)
print(len(rows))
print(reader.fieldnames)

## Marginal Distribution of Cumulative Spend

The marginal distribution is bounded between 0 and 1 and is strongly affected by repeated observations from the same item curve. It is not a standalone time-free distribution: a point at 90% elapsed and a point at 20% elapsed should not be expected to have the same cumulative spend behavior. The visible pile-up near 100% is expected because every completed item contributes a final cluster at full cumulative spend.

| Quantile | Cumulative spend pct |
| --- | --- |
| p01 | 0.87% |
| p05 | 6.48% |
| p10 | 14.12% |
| p25 | 35.00% |
| p50 | 66.69% |
| p75 | 94.54% |
| p90 | 100.00% |
| p95 | 100.00% |
| p99 | 100.00% |



## Joint Distribution With Elapsed Percent

The joint view is the core characterization. The distribution is bounded, monotonic in expectation, heteroscedastic, and edge-inflated near completion. The diagonal line is the pure linear spend reference; density above the line is front-loaded spend, while density below it is back-loaded or delayed spend.



## Conditional Distribution by Elapsed Bucket

The table and band plot summarize `CUMULATIVEBURNPCT | ELAPSEDPCT bucket`. The widening and narrowing of the percentile bands matters more than the overall histogram because the production question is where an item sits relative to expected cumulative spend at its current elapsed position.

| Elapsed bucket | Rows | Mean | Median | P10 | P25 | P75 | P90 | IQR | Skew |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 0.0-0.1 | 556 | 18.14% | 12.69% | 1.84% | 5.31% | 25.00% | 38.88% | 19.69% | 2.111 |
| 0.1-0.2 | 726 | 34.34% | 29.07% | 6.80% | 15.50% | 47.88% | 71.55% | 32.37% | 0.852 |
| 0.2-0.3 | 709 | 45.48% | 41.47% | 14.38% | 26.00% | 61.56% | 86.63% | 35.56% | 0.460 |
| 0.3-0.4 | 637 | 51.13% | 50.00% | 20.19% | 34.62% | 68.25% | 86.68% | 33.63% | 0.079 |
| 0.4-0.5 | 569 | 60.25% | 61.00% | 27.04% | 44.00% | 78.72% | 95.00% | 34.72% | -0.220 |
| 0.5-0.6 | 546 | 65.55% | 68.37% | 30.62% | 50.84% | 84.93% | 97.41% | 34.09% | -0.560 |
| 0.6-0.7 | 487 | 71.79% | 75.00% | 42.35% | 59.70% | 88.00% | 98.20% | 28.30% | -0.803 |
| 0.7-0.8 | 479 | 78.12% | 83.27% | 49.34% | 68.46% | 93.82% | 98.51% | 25.36% | -1.289 |
| 0.8-0.9 | 458 | 85.65% | 90.08% | 63.95% | 79.68% | 96.99% | 99.65% | 17.32% | -1.740 |
| 0.9-1.0 | 1,047 | 97.58% | 100.00% | 92.11% | 99.22% | 100.00% | 100.00% | 0.78% | -4.868 |

## Percentile Bands and Reference Curves

The empirical median curve is generally above the linear reference for much of the item life, which means these clustered payment curves are often front-loaded relative to time. The Beta CDF and anchored polynomial are compact parameterizations of that empirical shape; the later model notebook evaluates the Beta CDF approach more formally.



## Bucketed Density View

This view shows the full shape within each elapsed bucket rather than only percentiles. It makes the conditional spread visible: some elapsed regions are broad and multi-modal because items can receive lumpy clustered payments at different points in their lifecycle.



## Curve Parameterizations

| Model | Alpha / coefficients | Beta | MAE | RMSE | Bias | Clip share | Monotonic violations |
| --- | --- | --- | --- | --- | --- | --- | --- |
| Beta CDF | 0.486907 | 0.742934 | 0.1518 | 0.2077 | 0.0013 |  |  |
| Anchored polynomial degree 3 | 1.010083, 1.568445, -2.040373 |  | 0.1512 | 0.2082 | -0.0049 | 0.0080 | 0 |
| Anchored polynomial degree 4 | 0.992847, 2.086968, -4.523688, 2.678695 |  | 0.1517 | 0.2074 | -0.0009 | 0.0000 | 0 |

The Beta CDF is a useful production candidate because it is naturally bounded and monotone. The anchored polynomial is useful as a flexible descriptive reference, but it is less constrained structurally and should be treated as diagnostic unless monotonicity and stability are explicitly checked.

## Residual Distribution Around Reference Curves

Residuals are `actual cumulative pct - expected cumulative pct`. A good reference curve centers this distribution closer to zero and reduces asymmetric bias. The residual plot shows why the cumulative spend model should not be purely linear: the linear reference leaves a larger systematic position offset where the empirical spend curve is front-loaded.



## Stratification by Duration

Duration buckets have visibly different median cumulative spend curves. This supports the later duration-bucket Beta CDF model: the expected curve is not fully universal across short and long items.



## Stratification by Cluster Count

Cluster count is another proxy for payment cadence and lumpiness. Items with fewer clusters tend to show coarser jumps, while higher-cluster items provide smoother cumulative curves.



## Interpretation

The cumulative spend distribution is best understood as a conditional bounded distribution, not as a single ordinary marginal distribution. Its important properties are:

- Bounded support at `[0, 1]`, with a structural edge at 100% from final clusters.
- Strong positive relationship between elapsed percent and cumulative spend percent.
- A median curve that is not purely linear, indicating systematic front-loading in the clustered payment data.
- Wide conditional dispersion, especially in the middle of the lifecycle, caused by lumpy payment postings and differing payment cadence.
- Meaningful stratification by item duration and cluster count.

This motivates the next-stage notebook: fit expected-position curves and evaluate whether Beta CDF parameterizations outperform a transparent linear reference.